# Online Retail — Customer Segment Classification
**Dataset:** UCI Online Retail Dataset  
**Goal:** Classify customers into Low / Medium / High spenders based on past behavior  
**Models:** Logistic Regression, SVM (7 & 13 features), Calibrated SVM

### Design
- Features built from **past transactions (before Jun 2011)**
- Target: which spending segment after Jun 2011
- Segments defined by equal thirds (33/33/33) of future spend

---

## 0. Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV, CalibrationDisplay
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    f1_score, precision_score, recall_score
)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

---
## 1. Load & Clean Data

In [ ]:
df = pd.read_excel('Online Retail.xlsx', dtype={'CustomerID': str})
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]
df = df.dropna(subset=['CustomerID', 'Description'])
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]
df['Revenue']     = df['Quantity'] * df['UnitPrice']
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print(f'Clean dataset: {df.shape[0]:,} rows')
df.head()

---
## 2. Temporal Split

In [ ]:
cutoff_date = pd.Timestamp('2011-06-01')
past   = df[df['InvoiceDate'] <  cutoff_date].copy()
future = df[df['InvoiceDate'] >= cutoff_date].copy()
print(f'Past: {len(past):,}  Future: {len(future):,}')

---
## 3. Feature Engineering

In [ ]:
# Core RFM + order features
past_features = past.groupby('CustomerID').agg(
    Recency         = ('InvoiceDate', lambda x: (cutoff_date - x.max()).days),
    Frequency       = ('InvoiceNo',   'nunique'),
    Monetary_Past   = ('Revenue',     'sum'),
    Unique_Products = ('StockCode',   'nunique'),
    Max_Order       = ('Revenue',     'max'),
    Std_Order       = ('Revenue',     'std'),
    Avg_Items       = ('Quantity',    'mean'),
).reset_index()
past_features['Std_Order'] = past_features['Std_Order'].fillna(0)

# Tenure
tenure = past.groupby('CustomerID').agg(
    First_Purchase=('InvoiceDate','min'),
    Last_Purchase =('InvoiceDate','max')
).reset_index()
tenure['Tenure'] = (tenure['Last_Purchase'] - tenure['First_Purchase']).dt.days
tenure = tenure[['CustomerID','Tenure']]

# Weekend & Morning shopping patterns
past['Is_Weekend'] = past['InvoiceDate'].dt.dayofweek >= 5
past['Is_Morning'] = past['InvoiceDate'].dt.hour < 12
time_feats = past.groupby('CustomerID').agg(
    Pct_Weekend=('Is_Weekend','mean'),
    Pct_Morning=('Is_Morning','mean')
).reset_index()

# Product category diversity
past['Category'] = past['StockCode'].astype(str).str[:2]
categories = past.groupby('CustomerID').agg(
    Unique_Categories=('Category','nunique'),
    Top_Category_Pct =('Category', lambda x: x.value_counts().iloc[0]/len(x))
).reset_index()

# Active weeks
past['Week'] = past['InvoiceDate'].dt.isocalendar().week.astype(int)
active_weeks = past.groupby('CustomerID').agg(
    Active_Weeks=('Week','nunique')
).reset_index()

# Future target
future_target = (future.groupby('CustomerID')['Revenue']
                       .sum().reset_index()
                       .rename(columns={'Revenue':'Future_Monetary'}))

# Merge all
model_df = past_features.copy()
for fdf in [tenure, time_feats, categories, active_weeks, future_target]:
    model_df = model_df.merge(fdf, on='CustomerID', how='left')
model_df = model_df[model_df['Future_Monetary'] > 0].copy()

for col in ['Tenure','Pct_Weekend','Pct_Morning','Unique_Categories','Active_Weeks']:
    model_df[col] = model_df[col].fillna(0)
model_df['Top_Category_Pct'] = model_df['Top_Category_Pct'].fillna(1)

for col in ['Recency','Frequency','Monetary_Past','Unique_Products',
            'Max_Order','Std_Order','Avg_Items','Future_Monetary',
            'Tenure','Unique_Categories','Active_Weeks']:
    model_df[f'Log_{col}'] = np.log1p(model_df[col])

print(f'Final dataset: {model_df.shape[0]:,} customers, {model_df.shape[1]} columns')

---
## 4. Create Target Segments

In [ ]:
model_df['Segment'] = pd.qcut(
    model_df['Future_Monetary'], q=3, labels=[0,1,2]
).astype(int)

counts = model_df['Segment'].value_counts().sort_index()
for seg, label in zip([0,1,2],['Low','Medium','High']):
    print(f'  {label:8} (class {seg}): {counts[seg]:,} ({counts[seg]/len(model_df)*100:.1f}%)')

print('\nSpend ranges:')
print(model_df.groupby('Segment')['Future_Monetary'].agg(['min','max','mean']).round(2))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, (col, title) in enumerate([
    ('Future_Monetary','Future Spend by Segment'),
    ('Frequency',      'Frequency by Segment'),
    ('Recency',        'Recency by Segment'),
]):
    for seg, lbl in zip([0,1,2],['Low','Medium','High']):
        axes[i].hist(model_df[model_df['Segment']==seg][col],
                     bins=40, alpha=0.5, label=lbl, edgecolor='white')
    axes[i].set_title(title)
    axes[i].set_xlabel(col)
    axes[i].legend()
plt.suptitle('Feature Distributions by Segment', fontsize=13)
plt.tight_layout()
plt.show()

---
## 5. Correlation Analysis

In [ ]:
features_13 = [
    'Log_Recency','Log_Frequency','Log_Monetary_Past',
    'Log_Unique_Products','Log_Max_Order','Log_Std_Order','Log_Avg_Items',
    'Log_Tenure','Pct_Weekend','Pct_Morning',
    'Log_Unique_Categories','Top_Category_Pct','Log_Active_Weeks'
]

corr_df = model_df[features_13+['Segment']].corr()[['Segment']].drop('Segment')
corr_df = corr_df.sort_values('Segment', ascending=False)

plt.figure(figsize=(5, 8))
sns.heatmap(corr_df, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5)
plt.title('Feature Correlation with Segment')
plt.tight_layout()
plt.show()

---
## 6. Train/Test Split & Scaling

In [ ]:
features_7 = [
    'Log_Recency','Log_Frequency','Log_Monetary_Past',
    'Log_Unique_Products','Log_Max_Order','Log_Std_Order','Log_Avg_Items'
]
y = model_df['Segment']

X_13 = model_df[features_13]
X_train, X_test, y_train, y_test = train_test_split(
    X_13, y, test_size=0.2, random_state=42, stratify=y
)
scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'Train: {X_train.shape[0]:,}  Test: {X_test.shape[0]:,}')

---
## 7. Model Training — LR & SVM

In [ ]:
all_results = []
all_preds   = {}

for label, feats, clf in [
    ('LR — 7ft',   features_7,  LogisticRegression(max_iter=1000, random_state=42)),
    ('LR — 13ft',  features_13, LogisticRegression(max_iter=1000, random_state=42)),
    ('SVM — 7ft',  features_7,  SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)),
    ('SVM — 13ft', features_13, SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)),
]:
    X = model_df[feats]
    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    sc = StandardScaler()
    clf.fit(sc.fit_transform(Xtr), ytr)
    yp = clf.predict(sc.transform(Xte))
    all_results.append({
        'Model'    : label,
        'Accuracy' : accuracy_score(yte, yp),
        'F1'       : f1_score(yte, yp, average='weighted'),
        'Precision': precision_score(yte, yp, average='weighted'),
        'Recall'   : recall_score(yte, yp, average='weighted')
    })
    all_preds[label] = (yte, yp)
    print(f'{label:12}  Accuracy={accuracy_score(yte,yp):.4f}  '
          f'F1={f1_score(yte,yp,average="weighted"):.4f}')

---
## 8. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 11))
axes = axes.flatten()
for i, (label, (yte, yp)) in enumerate(all_preds.items()):
    cm   = confusion_matrix(yte, yp)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Low','Medium','High'])
    disp.plot(ax=axes[i], colorbar=False, cmap='Blues')
    axes[i].set_title(f'{label}  |  Accuracy: {accuracy_score(yte,yp):.4f}')
plt.suptitle('Confusion Matrices — All Models', fontsize=13)
plt.tight_layout()
plt.show()

---
## 9. Per-class Performance

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, metric in enumerate(['precision','recall','f1-score']):
    for j, (label, (yte, yp)) in enumerate(all_preds.items()):
        report = classification_report(yte, yp, output_dict=True)
        scores = [report[str(k)][metric] for k in range(3)]
        xpos   = np.arange(3) + j*0.2
        axes[i].bar(xpos, scores, 0.18, label=label,
                    color=['steelblue','coral','mediumseagreen','mediumpurple'][j],
                    edgecolor='white')
    axes[i].set_title(f'{metric.capitalize()} by Segment')
    axes[i].set_xticks(np.arange(3) + 0.3)
    axes[i].set_xticklabels(['Low','Medium','High'])
    axes[i].set_ylim(0, 1.0)
    axes[i].legend(fontsize=7)
plt.suptitle('Per-class Performance — All Models', fontsize=13)
plt.tight_layout()
plt.show()

---
## 10. SVM Calibration

SVM does not output real probabilities by default — it outputs decision scores.
Calibration converts these scores into trustworthy probabilities.

Two methods:
- **Sigmoid** (Platt scaling) — fits a logistic curve on top of SVM scores
- **Isotonic** — fits a flexible step function, can overfit on small datasets

In [ ]:
svm_base = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
svm_base.fit(X_train_s, y_train)

svm_sigmoid = CalibratedClassifierCV(
    SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42),
    method='sigmoid', cv=5
)
svm_sigmoid.fit(X_train_s, y_train)

svm_isotonic = CalibratedClassifierCV(
    SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42),
    method='isotonic', cv=5
)
svm_isotonic.fit(X_train_s, y_train)

print('--- Accuracy after calibration ---')
for label, model in [
    ('SVM — no calibration',       svm_base),
    ('SVM — sigmoid calibration',  svm_sigmoid),
    ('SVM — isotonic calibration', svm_isotonic),
]:
    yp = model.predict(X_test_s)
    print(f'  {label:35} '
          f'Accuracy={accuracy_score(y_test,yp):.4f}  '
          f'F1={f1_score(y_test,yp,average="weighted"):.4f}')

In [ ]:
# Calibration curves — closer to diagonal = better calibrated
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
class_labels = ['Low','Medium','High']

for i, (ax, class_label) in enumerate(zip(axes, class_labels)):
    y_binary = (y_test == i).astype(int)
    for label, model, color, ls in [
        ('Sigmoid',  svm_sigmoid,  'steelblue',     '-'),
        ('Isotonic', svm_isotonic, 'mediumseagreen', '--'),
    ]:
        probs = model.predict_proba(X_test_s)[:, i]
        CalibrationDisplay.from_predictions(
            y_binary, probs, n_bins=8, ax=ax,
            name=label, color=color, linestyle=ls
        )
    ax.set_title(f'Calibration Curve — {class_label} segment')
    ax.legend(fontsize=9)
plt.suptitle('Calibration Curves by Segment\n(diagonal = perfect calibration)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Probability distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, (ax, class_label) in enumerate(zip(axes, class_labels)):
    for label, model, color in [
        ('Sigmoid',  svm_sigmoid,  'steelblue'),
        ('Isotonic', svm_isotonic, 'coral'),
    ]:
        probs = model.predict_proba(X_test_s)[:, i]
        ax.hist(probs, bins=30, alpha=0.5, color=color,
                label=label, edgecolor='white')
    ax.set_title(f'Predicted Probabilities — {class_label}')
    ax.set_xlabel('Predicted probability')
    ax.set_ylabel('Count')
    ax.legend()
plt.suptitle('Probability Distributions — Sigmoid vs Isotonic', fontsize=13)
plt.tight_layout()
plt.show()

---
## 11. Precision@K — Business Impact

Rank customers by predicted probability of being a High spender.
Precision@K answers: if we target the top K customers, what % are actually High spenders?
This is the metric marketing teams care about most.

In [ ]:
ranking = X_test.copy().reset_index(drop=True)
ranking['True_Segment']       = y_test.values
ranking['Prob_High_Sigmoid']  = svm_sigmoid.predict_proba(X_test_s)[:, 2]
ranking['Prob_High_Isotonic'] = svm_isotonic.predict_proba(X_test_s)[:, 2]
ranking['Predicted_Segment']  = svm_isotonic.predict(X_test_s)
ranking = ranking.sort_values('Prob_High_Isotonic', ascending=False)

print('--- Top 15 customers most likely to be High spenders ---')
print(ranking[['True_Segment','Predicted_Segment',
               'Prob_High_Sigmoid','Prob_High_Isotonic']].head(15).to_string())

print('\n--- Precision at Top K (Isotonic) ---')
for k in [10, 20, 30, 50, 100]:
    precision = (ranking.head(k)['True_Segment'] == 2).sum() / k
    print(f'  Top {k:3d}: {precision:.1%} are actually High spenders')

In [ ]:
precisions = []
for k in range(1, len(ranking)+1):
    precisions.append((ranking.head(k)['True_Segment'] == 2).sum() / k)

plt.figure(figsize=(10, 5))
plt.plot(range(1, len(ranking)+1), precisions, color='steelblue', linewidth=2)
plt.axhline(1/3, color='red', linestyle='--', linewidth=1, label='Random baseline (0.33)')
plt.fill_between(range(1, len(ranking)+1), precisions, 1/3,
                 where=[p > 1/3 for p in precisions],
                 alpha=0.1, color='steelblue', label='Improvement over random')
plt.title('Precision@K — How accurately can we target High spenders?')
plt.xlabel('Number of customers targeted (K)')
plt.ylabel('Precision (% actually High spenders)')
plt.legend()
plt.tight_layout()
plt.show()

---
## 12. Final Model Comparison

In [ ]:
yp_sig = svm_sigmoid.predict(X_test_s)
yp_iso = svm_isotonic.predict(X_test_s)

final_results = pd.DataFrame([
    {'Model': 'LR — 7ft',       'Accuracy': 0.567, 'F1': 0.561},
    {'Model': 'SVM — 7ft',      'Accuracy': 0.573, 'F1': 0.573},
    {'Model': 'SVM — 13ft',     'Accuracy': 0.583, 'F1': 0.581},
    {'Model': 'SVM — Sigmoid',  'Accuracy': accuracy_score(y_test, yp_sig),
                                 'F1': f1_score(y_test, yp_sig, average='weighted')},
    {'Model': 'SVM — Isotonic', 'Accuracy': accuracy_score(y_test, yp_iso),
                                 'F1': f1_score(y_test, yp_iso, average='weighted')},
])

print(final_results.to_string(index=False))

x, width = np.arange(len(final_results)), 0.35
fig, ax  = plt.subplots(figsize=(12, 5))
b1 = ax.bar(x - width/2, final_results['Accuracy'], width,
            label='Accuracy', color='steelblue', edgecolor='white')
b2 = ax.bar(x + width/2, final_results['F1'], width,
            label='F1', color='coral', edgecolor='white')
for bars in [b1, b2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.004,
                f'{bar.get_height():.3f}',
                ha='center', fontsize=9, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(final_results['Model'], fontsize=10)
ax.set_ylim(0, 0.80)
ax.axhline(0.60, color='black', linestyle='--', linewidth=1, label='0.60 reference')
ax.set_title('Final Model Comparison — Including Calibrated SVM')
ax.set_ylabel('Score')
ax.legend()
plt.tight_layout()
plt.show()

---
## 13. Key Findings

### Verified results

| Model | Accuracy | F1 |
|---|---|---|
| LR — 7 features | 0.567 | 0.561 |
| SVM — 7 features | 0.573 | 0.573 |
| SVM — 13 features | 0.583 | 0.581 |
| SVM — Sigmoid calibrated | **0.586** | 0.577 |
| SVM — Isotonic calibrated | 0.583 | **0.582** |

### Precision@K — headline business result

| Target top K | % actually High spenders |
|---|---|
| Top 10 | 90% |
| Top 20 | 85% |
| Top 50 | 82% |
| Top 100 | 70% |
| Random baseline | 33% |

Targeting the top 20 customers is **2.6x more precise than random**.

### Why calibration matters
- SVM outputs decision scores, not real probabilities by default
- Without calibration, a score of 0.8 does not mean 80% confidence
- Sigmoid calibration improved accuracy and made probabilities trustworthy
- Isotonic is more flexible but can overfit on small datasets

### Natural ceiling ~0.58
- Only 6 months of past history per customer
- Medium segment is hardest — soft boundaries make it ambiguous
- Random Forest or XGBoost would be the recommended next step